Fáza 2 - Predspracovanie údajov

In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    PowerTransformer,
    QuantileTransformer,
    Normalizer,
    LabelEncoder
)
from sklearn.feature_selection import (
    SelectKBest,
    f_regression,
    mutual_info_regression,
    VarianceThreshold
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from pathlib import Path

path = Path("../data/cleaned_data")

observation_df = pd.read_csv(path / "observation.csv")
patient_df = pd.read_csv(path / "patient.csv")
station_df = pd.read_csv(path / "station.csv")

In [81]:
station_df["revision"] = pd.to_datetime(station_df["revision"], errors="coerce")
station_df["revision"] = station_df["revision"].astype("int64") // 10**9  

le = LabelEncoder()
for col in ["station", "continent", "city"]:
    if col in station_df.columns:
        station_df[col] = station_df[col].fillna("Missing")
        station_df[col] = le.fit_transform(station_df[col].astype(str))

station_df = pd.get_dummies(station_df, columns=["QoS"], prefix="QoS")

display(station_df.head())

,station,longitude,latitude,revision,continent,city,QoS_acceptable,QoS_excellent,QoS_good,QoS_maintenance
0,224,86.51499,23.19590,1529193600,4,57,False,False,True,False
1,72,-83.48216,42.30865,1665360000,1,32,False,False,True,False
2,513,84.87144,47.46657,1478390400,4,4,False,False,True,False
3,415,30.38167,59.80917,1662768000,6,81,False,False,True,False
4,85,2.95924,36.76775,1704585600,0,3,False,False,True,False


In [82]:
merged_df = pd.merge(
    observation_df,
    station_df,
    on=["latitude", "longitude"],
    how="left"
)

print("Non-null station data:", merged_df['station'].notna().mean() * 100, "%")

Non-null station data: 100.0 %


In [83]:
X = merged_df.drop('oximetry', axis=1)
y = merged_df['oximetry']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(17260, 30) (4315, 30)


In [84]:
print(merged_df.select_dtypes(include='object').columns)
print(merged_df.head())

Index([], dtype='object')
        SpO₂         HR         PI         RR      EtCO₂       FiO₂  \
0  96.491498  75.410035  11.966950  14.781380  38.814201  60.447244   
1  97.657620  90.314511   9.500074  16.118786  42.528828  60.183638   
2  96.528162  79.127147  11.268005  15.921648  41.262696  76.905716   
3  98.195660  83.015909  14.140633  14.443310  40.283494  59.027420   
4  97.573527  83.388999   9.058458  17.513347  39.844401  49.965222   

          PRV          BP  Skin Temperature  Motion/Activity index  ...  \
0  130.452955   97.686382         34.718865               8.254118  ...   
1  106.651627  107.891610         36.720399               9.514584  ...   
2  131.520657  109.488301         36.720399              10.435298  ...   
3  144.702141  101.915115         34.855279               8.896198  ...   
4  111.977370   98.441554         36.720399               7.009457  ...   

   latitude  longitude  station    revision  continent  city  QoS_acceptable  \
0   6.84019   79

In [85]:
X = merged_df.drop(columns=['oximetry'])
y = merged_df['oximetry']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scalers = {
    "StandardScaler": StandardScaler(),
    "RobustScaler": RobustScaler(),
    "PowerTransformer": PowerTransformer(),
    "QuantileTransformer": QuantileTransformer(output_distribution="uniform")
}

model = RandomForestRegressor(random_state=42)

results = []

for name, scaler in scalers.items():
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    results.append({
        "Scaler": name,
        "R2 Score": r2,
        "MAE": mae
    })

results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
display(results_df)

,Scaler,R2 Score,MAE
2,PowerTransformer,0.880462,0.081752
1,RobustScaler,0.880325,0.081539
0,StandardScaler,0.880321,0.081541
3,QuantileTransformer,0.880249,0.081638


In [86]:
X = merged_df.drop(columns=['oximetry'])
y = merged_df['oximetry']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scalers = {
    "StandardScaler": StandardScaler(),
    "RobustScaler": RobustScaler(),
    "PowerTransformer": PowerTransformer(),
    "QuantileTransformer": QuantileTransformer(output_distribution="uniform")
}

selectors = {
    "SelectKBest": SelectKBest(score_func=f_regression, k=10),
    "MutualInfo": SelectKBest(score_func=mutual_info_regression, k=10),
    "VarianceThreshold": VarianceThreshold(0.01)
}

model = RandomForestRegressor(random_state=42, n_estimators=100)

results = []

for scaler_name, scaler in scalers.items():
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    for selector_name, selector in selectors.items():
        try:
            X_train_sel = selector.fit_transform(X_train_scaled, y_train)
            X_test_sel = selector.transform(X_test_scaled)

            model.fit(X_train_sel, y_train)
            y_pred = model.predict(X_test_sel)

            r2 = r2_score(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)

            results.append({
                "Scaler": scaler_name,
                "Selector": selector_name,
                "R2 Score": r2,
                "MAE": mae
            })
        except Exception as e:
            print(f"{scaler_name} + {selector_name} failed: {e}")

results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
display(results_df)

,Scaler,Selector,R2 Score,MAE
0,StandardScaler,SelectKBest,0.899773,0.069404
3,RobustScaler,SelectKBest,0.899757,0.069402
6,PowerTransformer,SelectKBest,0.899629,0.069096
9,QuantileTransformer,SelectKBest,0.898892,0.069611
5,RobustScaler,VarianceThreshold,0.880325,0.081539
2,StandardScaler,VarianceThreshold,0.880321,0.081541
11,QuantileTransformer,VarianceThreshold,0.880193,0.081606
10,QuantileTransformer,MutualInfo,0.878763,0.078171
1,StandardScaler,MutualInfo,0.878627,0.078146
4,RobustScaler,MutualInfo,0.878627,0.078146


In [87]:
X = merged_df.drop(columns=['oximetry'])
y = merged_df['oximetry']

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('select', SelectKBest(score_func=f_regression, k='all')),
    ('model', LinearRegression())
])

pipeline.fit(X, y)

selector = pipeline.named_steps['select']
mask = selector.get_support()
selected_features = X.columns[mask]

coef = pipeline.named_steps['model'].coef_

coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': coef,
    'Abs_Coeff': abs(coef)
}).sort_values(by='Abs_Coeff', ascending=False)

display(coef_df.head(100))

,Feature,Coefficient,Abs_Coeff
10,PVI,0.261145,0.261145
6,PRV,0.095775,0.095775
1,HR,0.094190,0.094190
4,EtCO₂,-0.051431,0.051431
0,SpO₂,-0.023435,0.023435
25,city,0.008085,0.008085
24,continent,-0.007633,0.007633
18,O₂ extraction ratio,0.006577,0.006577
8,Skin Temperature,0.004596,0.004596
23,revision,0.004319,0.004319
